# Delocalization Score Pipeline — Example Workflow

This notebook walks through the **end-to-end pipeline** for computing delocalization scores from MALDI-MSI data.

Each step corresponds to one of the scripts in the `src/` directory. Steps 1, 2, and 4 run headlessly; **Steps 3 and 5 require a display** (Qt5 backend) for interactive GUIs.

> **Tip:** In practice, you'll likely run each step as a standalone script rather than inside this notebook — especially the interactive steps. This notebook is provided as a **reference guide** showing the order, inputs, outputs, and key parameters.

---

### Prerequisites

```
pip install -r requirements.txt
```

### Directory Layout

```
Delocalization_Score/
├── src/                          # Pipeline scripts
├── data/
│   ├── imzml_data/               # Place .imzml + .ibd files here
│   ├── H&E_images/               # Place H&E images here
│   ├── raw_h5ad/                  # Created by Step 1
│   ├── processed_h5ad/            # Created by Step 2
│   ├── overlaid_h5ad/             # Created by Step 3
│   └── delocalization_score_result_h5ad/  # Created by Step 4
├── notebooks/
│   └── example_workflow.ipynb     # ← You are here
└── ...
```

---
## Step 1 — imzML to AnnData Conversion

**Script:** `src/1_imzml_reader.py`

Converts raw `.imzml` / `.ibd` files into AnnData (`.h5ad`) objects with:
- Sparse (CSR) intensity matrix — pixels × m/z bins
- Spatial coordinates in `obs["x"]`, `obs["y"]` and `obsm["spatial"]`
- TIC-normalized intensities in `layers["tic_normalized"]`
- Metadata columns: sample, batch, age group, disease status

| Parameter | Description | Example |
|---|---|---|
| `input_filename` | imzML file in `imzml_data/` | `"sample_1.imzml"` |
| `output_filename` | Output `.h5ad` in `raw_h5ad/` | `"sample_1.h5ad"` |
| `mz_decimals` | Decimal places for m/z rounding | `6` |
| `sample`, `batch`, `age_group`, `disease_status` | Metadata labels | `"1-1"`, `"Slide_1"`, `"Aged"`, `"AD"` |

In [ ]:
# To run Step 1, edit the config block in the script and execute:
# !python ../src/1_imzml_reader.py

# Or configure and call from here:
import os
WORKING_DIR = os.path.abspath("../data")
INPUT_FILE  = os.path.join(WORKING_DIR, "imzml_data", "sample_1.imzml")
OUTPUT_FILE = os.path.join(WORKING_DIR, "raw_h5ad", "sample_1.h5ad")

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print("\n→ Run: python ../src/1_imzml_reader.py")

---
## Step 2 — Peak Detection & Lock-Mass Correction

**Script:** `src/2_peak_detection.py`

Processes the raw AnnData through:
1. **Asymmetric Least Squares (ALS)** baseline correction
2. **Savitzky–Golay** smoothing
3. **Peak picking** with `scipy.signal.find_peaks` + proximity filtering
4. **Lock-mass polynomial correction** for m/z drift

| Parameter | Description | Default |
|---|---|---|
| `sg_window_length` | Savitzky–Golay window | `9` |
| `sg_polyorder` | Savitzky–Golay polynomial order | `3` |
| `peak_height_percentile` | Min height as percentile of max | `10` |
| `min_peak_distance` | Minimum bins between peaks | `5` |
| `proximity_threshold` | Merge peaks within this m/z gap | `0.5` |
| `top_n_peaks` | Number of peaks to retain | `1000` |
| `lock_mass_references` | Known reference masses for correction | `{798.541: 798.541, ...}` |
| `lock_mass_poly_degree` | Polynomial degree for drift fit | `2` |

In [ ]:
# To run Step 2:
# !python ../src/2_peak_detection.py

INPUT_FILE  = os.path.join(WORKING_DIR, "raw_h5ad", "sample_1.h5ad")
OUTPUT_FILE = os.path.join(WORKING_DIR, "processed_h5ad", "sample_1_peaks.h5ad")

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print("\n→ Run: python ../src/2_peak_detection.py")

---
## Step 3 — H&E Overlay & Tissue Annotation

**Script:** `src/3_mz_overlay.py`

⚠️ **Requires a display** (Qt5 backend for matplotlib)

This interactive step has two phases:

### Phase A — Image Registration
A GUI with sliders lets you align the H&E histology image to an MSI ion heatmap by adjusting:
- X/Y offset, rotation, and scale

### Phase B — Tissue Annotation
Draw a **polygon ROI** on the aligned image to define tissue vs. background. The binary mask is stored in `obs["tissue"]` (1 = tissue, 0 = background).

| Parameter | Description | Example |
|---|---|---|
| `REFERENCE_MZ` | m/z value for the alignment heatmap | `798.541` |
| `MZ_TOLERANCE` | Tolerance for matching m/z columns | `0.5` |

In [ ]:
# Step 3 must be run as a standalone script (needs interactive GUI):
# !python ../src/3_mz_overlay.py

INPUT_H5AD = os.path.join(WORKING_DIR, "processed_h5ad", "sample_1_peaks.h5ad")
HE_IMAGE   = os.path.join(WORKING_DIR, "H&E_images", "sample_1_HE.jpg")
OUTPUT_FILE = os.path.join(WORKING_DIR, "overlaid_h5ad", "sample_1_overlaid.h5ad")

print(f"Input h5ad: {INPUT_H5AD}")
print(f"H&E image:  {HE_IMAGE}")
print(f"Output:     {OUTPUT_FILE}")
print("\n⚠️  Run in a terminal with display access:")
print("   python ../src/3_mz_overlay.py")

---
## Step 4 — Delocalization Score Calculation

**Script:** `src/4_delocalization_score_calculator.py`

Computes the composite delocalization score for every m/z feature. For each feature:

1. **Binarize** — threshold the intensity map (nonzero pixels)
2. **Nonzero off-tissue area** — count background pixels with signal, normalize by total background pixels
3. **Mean BG-to-tissue distance** — for each off-tissue pixel with signal, compute distance to nearest tissue pixel (KD-tree accelerated), normalize by the maximum possible distance
4. **Composite score:**

$$
D = \alpha \cdot \hat{A}_{\text{off}} + (1 - \alpha) \cdot \hat{d}_{\text{mean}}
$$

where $\alpha \in [0, 1]$ controls the weighting (default: 0.5).

Results are stored in `adata.var`:
- `nonzero_off_tissue_area` — raw count of off-tissue signal pixels
- `normalized_off_tissue_area` — normalized to [0, 1]
- `mean_bg_to_tissue_distance` — raw mean distance
- `normalized_mean_distance` — normalized to [0, 1]
- `delocalization_score` — final composite score

In [ ]:
# To run Step 4:
# !python ../src/4_delocalization_score_calculator.py

INPUT_FILE  = os.path.join(WORKING_DIR, "overlaid_h5ad", "sample_1_overlaid.h5ad")
OUTPUT_FILE = os.path.join(WORKING_DIR, "delocalization_score_result_h5ad", "sample_1_scored.h5ad")

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print("\n→ Run: python ../src/4_delocalization_score_calculator.py")

### Inspect the Results

After Step 4 completes, you can load the scored AnnData and inspect the delocalization scores:

In [ ]:
# Uncomment and run after Step 4 has completed:

# import anndata as ad
# import pandas as pd
#
# adata = ad.read_h5ad(OUTPUT_FILE)
#
# # Show top-10 most delocalized features
# scores = adata.var[["delocalization_score", "normalized_off_tissue_area", "normalized_mean_distance"]]
# scores = scores.sort_values("delocalization_score", ascending=False)
# print("Top 10 most delocalized m/z features:")
# print(scores.head(10).to_string())
#
# # Summary statistics
# print("\nDelocalization score summary:")
# print(scores["delocalization_score"].describe())

---
## Step 5 — Visualization

**Script:** `src/5_visualization_tools.py`

⚠️ **Requires a display** (Qt5 backend)

Generates three types of visualizations for a target m/z feature:

1. **Spatial heatmap** — ion intensity map with tissue contour overlay
2. **Distance arrow map** — arrows from each off-tissue signal pixel toward the nearest tissue pixel, color-coded by distance
3. **Score summary** — delocalization score and component metrics

| Parameter | Description | Example |
|---|---|---|
| `TARGET_MZ` | m/z feature to visualize | `798.541` |
| `MZ_TOLERANCE` | Tolerance for matching the target m/z | `0.5` |

In [ ]:
# Step 5 must be run as a standalone script (needs interactive GUI):
# !python ../src/5_visualization_tools.py

INPUT_FILE = os.path.join(WORKING_DIR, "delocalization_score_result_h5ad", "sample_1_scored.h5ad")

print(f"Input: {INPUT_FILE}")
print("\n⚠️  Run in a terminal with display access:")
print("   python ../src/5_visualization_tools.py")

---
## Batch Processing

To process multiple samples, loop over your sample list and update the configuration for each run. Here's a template:

In [ ]:
# Template for batch processing (Steps 1, 2, and 4 only — Step 3 is interactive)

samples = [
    {"filename": "sample_1", "sample": "1-1", "batch": "Slide_1", "age": "Aged",  "disease": "AD"},
    {"filename": "sample_2", "sample": "2-1", "batch": "Slide_1", "age": "Aged",  "disease": "WT"},
    {"filename": "sample_3", "sample": "3-1", "batch": "Slide_2", "age": "Young", "disease": "AD"},
]

for s in samples:
    print(f"Processing {s['filename']}...")
    print(f"  Step 1: {s['filename']}.imzml → {s['filename']}.h5ad")
    print(f"  Step 2: {s['filename']}.h5ad → {s['filename']}_peaks.h5ad")
    print(f"  Step 3: [interactive — run manually]")
    print(f"  Step 4: {s['filename']}_overlaid.h5ad → {s['filename']}_scored.h5ad")
    print()

---
## Citation

If you use this pipeline, please cite:

> Jarrahi, A., Jones, A., Tang, W., Qi, H., & Crouch, A. C. (2026). Mathematical Framework for Quantifying Delocalization in MALDI-MSI via a Composite Scoring Approach. *ACS Measurement Science Au*, 6(1), 134–149. https://doi.org/10.1021/acsmeasuresciau.5c00148

---

## Contact

- **Amin Jarrahi** — ajarrah1@vols.utk.edu
- **Dr. Anna Colleen Crouch** — acrouch5@utk.edu
- [Crouch Imaging Lab](https://crouchimaging.utk.edu) — University of Tennessee, Knoxville